# 03a — Unsupervised models: tuning on v1, the tuning log and the final score files

Owner: Arva (A9–A10, A15–A16). Isolation Forest and the autoencoder from `src/models_unsupervised.py` on `data/processed/features_v1.parquet`, tuned on the **val** split only (TEAM_PLAN 2.5.6: never tune on test). Every run adds one row to the tuning log `results/tables/tuning_log_unsupervised.csv`: parameters, val ROC-AUC, val PR-AUC, val recall when the top 1% of val user-days are flagged, and training time. The best run is the one with the highest **val PR-AUC**.

`DATASET` below picks the feature table. The v0 run (A9–A10, Nov 4) keeps its rows in the log: a full run replaces only the rows of its own dataset (`clear_log`).

Run with Restart & Run All (about half an hour on v1; the autoencoder grid takes most of it).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd

from src import models_unsupervised as mu
from src.config import PROCESSED_DIR, TABLES_DIR

DATASET = 'v1'
df = pd.read_parquet(PROCESSED_DIR / f'features_{DATASET}.parquet')
log_path = TABLES_DIR / mu.TUNING_LOG
mu.clear_log(DATASET)                  # drop only this dataset's old rows; other datasets stay (A15)
X, y, cols = mu.get_model_matrix(df, split='train')
print(f'{DATASET}: {len(df):,} user-days, {len(cols)} features, '
      f'{int(((df.split == "train") & (df.label == 0)).sum()):,} normal training rows')
df.groupby('split').label.agg(['size', 'sum']).rename(columns={'size': 'rows', 'sum': 'malicious'})

v1: 330,268 user-days, 76 features, 208,494 normal training rows


,rows,malicious
split,,
test,65007,170
train,209072,578
val,56189,236


## A9 / A15: reference run with the card settings

Isolation Forest (A5 card) and the autoencoder (A6 card), each scored on every row and saved as `results/scores/<model>_<DATASET>.parquet`. The card values are passed explicitly, because the module defaults are the tuned settings since A16. The final `<model>.parquet` files come from the last section.

In [2]:
first = []
for name in ('iforest', 'autoencoder'):
    card = mu.DETECTORS[name].card
    log, _ = mu.tune(df, name, grid={}, fixed=card, dataset=DATASET)   # an empty grid = one run
    scores, info = mu.train_and_score(df, name, card, save_as=f'{name}_{DATASET}')
    first.append(log.assign(saved=str(info['path'].relative_to(ROOT))))
    if name == 'autoencoder':
        fig = mu.plot_loss_curves(info['detector'].history, f'autoencoder_loss_{DATASET}')
pd.concat(first)[['model', 'params', 'val_roc_auc', 'val_pr_auc', 'val_recall_top1pct', 'fit_seconds', 'saved']].round(4)

,model,params,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds,saved
0,iforest,"{""n_estimators"": 200, ""max_samples"": ""auto"", ""...",0.8567,0.0147,0.0254,0.65,results\scores\iforest_v1.parquet
0,autoencoder,"{""hidden"": [64, 32], ""bottleneck"": 16, ""dropou...",0.8724,0.0916,0.1907,41.90,results\scores\autoencoder_v1.parquet


## A10: grid search

- **Isolation Forest:** `n_estimators` {100, 200, 500} × `max_samples` {256, 1024, auto} × `max_features` {0.5, 1.0} = 18 runs.
- **Autoencoder:** bottleneck {8, 16} × hidden layers {(64, 32), (128, 64)} × dropout {0, 0.1} × learning rate {1e-3, 1e-4} = 16 runs.

In [3]:
if_log, if_best = mu.tune(df, 'iforest', dataset=DATASET)
print('best Isolation Forest (val PR-AUC):', if_best)
if_log.sort_values('val_pr_auc', ascending=False).head(5).round(4)

best Isolation Forest (val PR-AUC): {'max_features': 1.0, 'max_samples': 256, 'n_estimators': 100}


,model,dataset,train_on,params,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds,n_train,n_features
15,iforest,v1,normal,"{""max_features"": 1.0, ""max_samples"": ""auto"", ""...",0.8643,0.0156,0.0169,0.38,208494,76
9,iforest,v1,normal,"{""max_features"": 1.0, ""max_samples"": 256, ""n_e...",0.8643,0.0156,0.0169,0.38,208494,76
13,iforest,v1,normal,"{""max_features"": 1.0, ""max_samples"": 1024, ""n_...",0.8522,0.0154,0.0254,0.86,208494,76
14,iforest,v1,normal,"{""max_features"": 1.0, ""max_samples"": 1024, ""n_...",0.8573,0.0153,0.0254,1.68,208494,76
7,iforest,v1,normal,"{""max_features"": 0.5, ""max_samples"": ""auto"", ""...",0.8551,0.0149,0.0297,0.94,208494,76


In [4]:
ae_log, ae_best = mu.tune(df, 'autoencoder', dataset=DATASET)
print('best autoencoder (val PR-AUC):', ae_best)
ae_log.sort_values('val_pr_auc', ascending=False).head(5).round(4)

best autoencoder (val PR-AUC): {'bottleneck': 16, 'dropout': 0.0, 'hidden': (128, 64), 'lr': 0.001}


,model,dataset,train_on,params,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds,n_train,n_features
10,autoencoder,v1,normal,"{""bottleneck"": 16, ""dropout"": 0.0, ""hidden"": [...",0.9185,0.1094,0.2246,17.21,208494,76
2,autoencoder,v1,normal,"{""bottleneck"": 8, ""dropout"": 0.0, ""hidden"": [1...",0.9058,0.1006,0.1695,50.76,208494,76
15,autoencoder,v1,normal,"{""bottleneck"": 16, ""dropout"": 0.1, ""hidden"": [...",0.9091,0.0939,0.1653,88.50,208494,76
14,autoencoder,v1,normal,"{""bottleneck"": 16, ""dropout"": 0.1, ""hidden"": [...",0.9056,0.0920,0.1568,30.60,208494,76
8,autoencoder,v1,normal,"{""bottleneck"": 16, ""dropout"": 0.0, ""hidden"": [...",0.8724,0.0916,0.1907,46.84,208494,76


## A15 step 5: contamination variants with the chosen settings

The same two models with their best settings, trained on **every** training row (malicious days included, `train_on='all'`). One log row each; `hyperparameter_table()` ignores them, so T4 shows the normal-days choice.

In [5]:
contam = [mu.tune(df, name, grid={}, fixed=best, dataset=DATASET, train_on='all')[0]
          for name, best in (('iforest', if_best), ('autoencoder', ae_best))]
pd.concat(contam)[['model', 'train_on', 'params', 'val_roc_auc', 'val_pr_auc', 'val_recall_top1pct', 'fit_seconds']].round(4)

,model,train_on,params,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds
0,iforest,all,"{""max_features"": 1.0, ""max_samples"": 256, ""n_e...",0.8375,0.0130,0.0169,0.44
0,autoencoder,all,"{""bottleneck"": 16, ""dropout"": 0.0, ""hidden"": [...",0.8972,0.0542,0.1144,69.86


## A13 (optional): One-Class SVM

`OneClassSVM(kernel='rbf', nu=0.01, gamma='scale')` in the same scaler pipeline, fitted on a random sample of at most 20,000 normal training rows (it scales poorly). Saved as `results/scores/ocsvm_<DATASET>.parquet`.

In [6]:
oc_log, _ = mu.tune(df, 'ocsvm', grid={}, dataset=DATASET)
scores, info = mu.train_and_score(df, 'ocsvm', save_as=f'ocsvm_{DATASET}')
print(f"fitted on {info['detector'].n_fitted:,} rows in {info['fit_seconds']:.1f}s; scoring {info['score_seconds']:.1f}s")
oc_log[['model', 'val_roc_auc', 'val_pr_auc', 'val_recall_top1pct', 'fit_seconds']].round(4)

fitted on 20,000 rows in 0.6s; scoring 10.8s


,model,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds
0,ocsvm,0.7748,0.0691,0.1229,0.61


## The tuning log so far

In [7]:
log = pd.read_csv(log_path)
print(f'{len(log)} runs in {log_path.relative_to(ROOT)}')
print(log.groupby(['dataset', 'model', 'train_on']).size().to_string())
runs = log[(log.dataset == DATASET) & (log.train_on == 'normal')]
best = runs.loc[runs.groupby('model').val_pr_auc.idxmax()]
best[['model', 'dataset', 'params', 'val_roc_auc', 'val_pr_auc', 'val_recall_top1pct', 'fit_seconds']].round(4)

76 runs in results\tables\tuning_log_unsupervised.csv
dataset  model        train_on
v0       autoencoder  normal      17
         iforest      normal      19
         ocsvm        normal       1
v1       autoencoder  all          1
                      normal      17
         iforest      all          1
                      normal      19
         ocsvm        normal       1


,model,dataset,params,val_roc_auc,val_pr_auc,val_recall_top1pct,fit_seconds
67,autoencoder,v1,"{""bottleneck"": 16, ""dropout"": 0.0, ""hidden"": [...",0.9185,0.1094,0.2246,17.21
48,iforest,v1,"{""max_features"": 1.0, ""max_samples"": 256, ""n_e...",0.8643,0.0156,0.0169,0.38
75,ocsvm,v1,{},0.7748,0.0691,0.1229,0.61


## Table T4 (A14 step 2, A15 step 3)

Card values, current module defaults, grid searched, and the best setting by val PR-AUC among this dataset's normal-days runs. Saved as `results/tables/T4_hyperparameters_unsupervised.csv`.

In [8]:
mu.hyperparameter_table(dataset=DATASET).round(4)

,model,card,defaults,grid,best_params,dataset,val_pr_auc,val_roc_auc,runs
0,iforest,"{""n_estimators"": 200, ""max_samples"": ""auto"", ""...","{""n_estimators"": 100, ""max_samples"": 256, ""max...","{""n_estimators"": [100, 200, 500], ""max_samples...","{""max_features"": 1.0, ""max_samples"": 256, ""n_e...",v1,0.0156,0.8643,19
1,autoencoder,"{""hidden"": [64, 32], ""bottleneck"": 16, ""dropou...","{""hidden"": [128, 64], ""bottleneck"": 16, ""dropo...","{""bottleneck"": [8, 16], ""hidden"": [[64, 32], [...","{""bottleneck"": 16, ""dropout"": 0.0, ""hidden"": [...",v1,0.1094,0.9185,17
2,ocsvm,"{""nu"": 0.01, ""gamma"": ""scale"", ""kernel"": ""rbf""...","{""nu"": 0.01, ""gamma"": ""scale"", ""kernel"": ""rbf""...",{},{},v1,0.0691,0.7748,1


## A16: final score files and `evaluate_file()` on each

The module defaults must equal the best v1 settings above (the first lines check it). `mu.final()` then writes `iforest`, `autoencoder`, `iforest_contam`, `autoencoder_contam` and `ocsvm` to `results/scores/` for every row, with no grid search; teammates run `python -m src.models_unsupervised final` to get the same files (TEAM_PLAN 2.5.12). `evaluate_file()` reports on the **test** split at both operating points.

In [9]:
from src.evaluate import evaluate_file

assert DATASET == 'v1', 'the final score files come from v1 only'
for name, best in (('iforest', if_best), ('autoencoder', ae_best)):
    defaults = mu.DETECTORS[name].defaults
    same = all(defaults[k] == v for k, v in best.items())
    print(f'{name}: module defaults == best v1 setting: {same}')
    assert same, f'{name}: commit {best} as the defaults in src/models_unsupervised.py (A16 step 2)'

rows = []
for path in mu.final(df=df):
    result = evaluate_file(path)
    row = {'model': path.stem, 'test_pr_auc': result['ranking']['pr_auc'],
           'test_roc_auc': result['ranking']['roc_auc']}
    for op in mu.OPERATING_POINTS:
        point = result['operating_points'][op]
        row.update({f'precision_{op}': point['precision'], f'recall_{op}': point['recall'],
                    f'user_recall_{op}': point['user_recall']})
    rows.append(row)
pd.DataFrame(rows).round(4)

iforest: module defaults == best v1 setting: True
autoencoder: module defaults == best v1 setting: True


iforest.parquet              330,268 rows scored, fit 0.4s on 208,494 rows


autoencoder.parquet          330,268 rows scored, fit 17.2s on 208,494 rows


iforest_contam.parquet       330,268 rows scored, fit 0.4s on 209,072 rows


autoencoder_contam.parquet   330,268 rows scored, fit 64.4s on 209,072 rows


ocsvm.parquet                330,268 rows scored, fit 0.6s on 208,494 rows


,model,test_pr_auc,test_roc_auc,precision_best_f1,recall_best_f1,user_recall_best_f1,precision_top_1pct,recall_top_1pct,user_recall_top_1pct
0,iforest,0.0122,0.8792,0.0117,0.5235,0.9333,0.0138,0.0529,0.2667
1,autoencoder,0.0833,0.9088,0.0698,0.1588,0.6000,0.0415,0.1588,0.6000
2,iforest_contam,0.0101,0.8606,0.0108,0.7118,1.0000,0.0138,0.0529,0.2000
3,autoencoder_contam,0.0402,0.9047,0.1429,0.0529,0.4000,0.0415,0.1588,0.6667
4,ocsvm,0.0574,0.7412,0.8182,0.0529,0.4000,0.0384,0.1471,0.6667


**Notes (Arva):** what the numbers say about v1 and why these settings were kept. *Human-only: write your own reading here.*